# 02 — Exploratory Data Analysis

> **Insiders Loyalty Program** · Customer Segmentation Project

---

Géron (Ch. 2): *"Explore the data to gain insights. You want to get a feeling for the data before designing your model."*

This notebook covers:
1. RFM feature distributions (after cleaning)
2. Revenue concentration — Pareto analysis
3. Customer behaviour hypotheses
4. Correlation matrix
5. Top customers by revenue

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from insiders_loyalty_program.config import load_config
from insiders_loyalty_program.data import load_training_frame
from insiders_loyalty_program.features import build_rfm_features

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('tab10')
FIGURES = PROJECT_ROOT / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

config = load_config(PROJECT_ROOT / 'configs' / 'project.toml')
print('Ready.')

In [ ]:
# Load raw transactions and build RFM table (one row per customer)
df_raw = load_training_frame(config)
rfm = build_rfm_features(df_raw, config)
print(f'Customers in RFM table: {len(rfm):,}')
rfm.head()

## 1. RFM Feature Distributions

Before modelling, we look at the distribution of each RFM feature. We expect **right-skewed** distributions — a small number of customers drive most of the value. This is the well-known **80/20 rule (Pareto Principle)** applied to retail.

In [ ]:
rfm_features = ['recency_days', 'frequency', 'monetary', 'avg_ticket', 'total_items']
rfm_features = [f for f in rfm_features if f in rfm.columns]

n_cols = min(3, len(rfm_features))
n_rows = (len(rfm_features) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for ax, feat in zip(axes, rfm_features):
    clip_val = rfm[feat].quantile(0.99)
    data = rfm[feat].clip(upper=clip_val)
    ax.hist(data, bins=40, color='steelblue', edgecolor='white', linewidth=0.4)
    ax.set_title(feat.replace('_', ' ').title())
    ax.set_xlabel(feat)
    ax.set_ylabel('Customers')
    mean_val = rfm[feat].median()
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=1.5, label=f'Median: {mean_val:.1f}')
    ax.legend(fontsize=8)

for ax in axes[len(rfm_features):]:
    ax.set_visible(False)

plt.suptitle('RFM Feature Distributions (clipped at 99th percentile)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES / '02_rfm_distributions.png', dpi=120)
plt.show()
print('All features are right-skewed. A log1p transform is needed before clustering.')

## 2. Pareto Analysis — Revenue Concentration

A core business hypothesis: **a small fraction of customers generates most of the revenue**.

If true, the Insiders program should focus on that top segment.

In [ ]:
if 'monetary' in rfm.columns:
    rfm_sorted = rfm.sort_values('monetary', ascending=False).reset_index(drop=True)
    rfm_sorted['cumulative_rev'] = rfm_sorted['monetary'].cumsum()
    rfm_sorted['pct_customers'] = (rfm_sorted.index + 1) / len(rfm_sorted) * 100
    rfm_sorted['pct_revenue']   = rfm_sorted['cumulative_rev'] / rfm_sorted['monetary'].sum() * 100

    # Find the customer % that accounts for 80% of revenue
    idx_80 = (rfm_sorted['pct_revenue'] >= 80).idxmax()
    pct_cust_for_80 = rfm_sorted.loc[idx_80, 'pct_customers']

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(rfm_sorted['pct_customers'], rfm_sorted['pct_revenue'],
            color='steelblue', linewidth=2)
    ax.axhline(80, color='red', linestyle='--', linewidth=1, label='80% of revenue')
    ax.axvline(pct_cust_for_80, color='orange', linestyle='--', linewidth=1,
               label=f'{pct_cust_for_80:.1f}% of customers')
    ax.fill_between(rfm_sorted['pct_customers'], rfm_sorted['pct_revenue'],
                    alpha=0.1, color='steelblue')
    ax.set_xlabel('% of Customers (ranked by revenue)')
    ax.set_ylabel('% Cumulative Revenue')
    ax.set_title('Pareto Curve — Revenue Concentration')
    ax.legend()
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    plt.tight_layout()
    plt.savefig(FIGURES / '02_pareto_curve.png', dpi=120)
    plt.show()

    print(f'\n>>> The top {pct_cust_for_80:.1f}% of customers account for 80% of total revenue.')
    print('This confirms the Pareto hypothesis and validates the Insiders program concept.')

## 3. Recency vs Frequency vs Monetary — 2D Scatter Plots

We use scatter plots (log scale) to see how customers are distributed across the RFM dimensions.

In [ ]:
if all(c in rfm.columns for c in ['recency_days', 'frequency', 'monetary']):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Recency vs Monetary
    axes[0].scatter(rfm['recency_days'], rfm['monetary'],
                    alpha=0.35, s=15, color='steelblue')
    axes[0].set_xscale('log')
    axes[0].set_yscale('log')
    axes[0].set_xlabel('Recency (days, log)')
    axes[0].set_ylabel('Monetary (£, log)')
    axes[0].set_title('Recency vs Monetary')

    # Frequency vs Monetary
    axes[1].scatter(rfm['frequency'], rfm['monetary'],
                    alpha=0.35, s=15, color='darkorange')
    axes[1].set_xscale('log')
    axes[1].set_yscale('log')
    axes[1].set_xlabel('Frequency (invoices, log)')
    axes[1].set_ylabel('Monetary (£, log)')
    axes[1].set_title('Frequency vs Monetary')

    plt.suptitle('Customer RFM Scatter (log scale)', fontsize=12)
    plt.tight_layout()
    plt.savefig(FIGURES / '02_rfm_scatter.png', dpi=120)
    plt.show()

## 4. Correlation Matrix

In [ ]:
numeric_rfm = rfm[rfm_features].select_dtypes(include='number')
corr = numeric_rfm.corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            ax=ax, linewidths=0.5, square=True)
ax.set_title('RFM Feature Correlation Matrix')
plt.tight_layout()
plt.savefig(FIGURES / '02_correlation_matrix.png', dpi=120)
plt.show()

## 5. Hypothesis Validation

Before modelling, we validate three business hypotheses.

In [ ]:
# H1: High-monetary customers are also recent buyers
if 'recency_days' in rfm.columns and 'monetary' in rfm.columns:
    top25_monetary = rfm['monetary'].quantile(0.75)
    top_customers   = rfm[rfm['monetary'] >= top25_monetary]
    rest_customers  = rfm[rfm['monetary'] <  top25_monetary]
    print('H1: Are high-monetary customers also recent buyers?')
    print(f'  Top 25% monetary — median recency: {top_customers["recency_days"].median():.0f} days')
    print(f'  Rest 75% monetary — median recency: {rest_customers["recency_days"].median():.0f} days')
    if top_customers['recency_days'].median() < rest_customers['recency_days'].median():
        print('  ✓ CONFIRMED: High-value customers buy more recently.')
    else:
        print('  ✗ NOT CONFIRMED: No clear recency advantage for high-value customers.')
    print()

In [ ]:
# H2: High avg_ticket does NOT necessarily mean high frequency
if 'avg_ticket' in rfm.columns and 'frequency' in rfm.columns:
    corr_val = rfm['avg_ticket'].corr(rfm['frequency'])
    print(f'H2: Correlation between avg_ticket and frequency: {corr_val:.3f}')
    if abs(corr_val) < 0.4:
        print('  ✓ CONFIRMED: Ticket size and frequency are largely independent.')
    else:
        print('  ✗ NOT CONFIRMED: Moderate or high correlation found.')
    print()

In [ ]:
# H3: A small cluster of customers drives most of the revenue
if 'monetary' in rfm.columns:
    top10_pct = rfm['monetary'].quantile(0.90)
    top10_rev = rfm[rfm['monetary'] >= top10_pct]['monetary'].sum()
    total_rev  = rfm['monetary'].sum()
    print(f'H3: Revenue share of top 10% customers: {top10_rev / total_rev * 100:.1f}%')
    if top10_rev / total_rev > 0.50:
        print('  ✓ CONFIRMED: Top 10% drives >50% of revenue — Insiders program is well justified.')
    else:
        print('  ✗ NOT CONFIRMED: Revenue is more evenly distributed.')

## 6. Top 20 Customers by Revenue

In [ ]:
if 'monetary' in rfm.columns:
    top20 = rfm.nlargest(20, 'monetary')[['customer_id', 'recency_days', 'frequency', 'monetary', 'avg_ticket']]
    top20 = top20.reset_index(drop=True)
    top20.index += 1
    top20['monetary'] = top20['monetary'].map('£{:,.0f}'.format)
    top20['avg_ticket'] = top20['avg_ticket'].map('£{:,.0f}'.format)
    print('Top 20 Customers by Total Revenue:')
    display(top20)

## 7. EDA Summary

| Finding | Implication |
|---|---|
| All RFM features are right-skewed | **Log transform** needed before clustering |
| Top ~20% of customers drive ~80% of revenue | Insiders program has strong business case |
| High monetary ≠ high frequency | Both dimensions needed for segmentation |
| Natural clusters exist in scatter plots | K-Means is a good fit for this problem |

---
**Next notebook:** `03_feature_engineering.ipynb` — build the preprocessing pipeline.